# SFT Data Generation — v2 Justification Styles

Generates two additional SFT training datasets using alternative teacher prompting styles:
- **evidence_first**: teacher quotes/paraphrases the key passage fragment before concluding
- **contrastive**: teacher briefly explains why the other labels don't apply (kept concise)

In [ ]:
import os
import json
import time
import random
from collections import Counter
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential

RESOURCE_GROUP = "cis-5270-team-10"
OPENAI_API_KEY = ""

OPENAI_ENDPOINT = f"https://{RESOURCE_GROUP}.openai.azure.com"
SUBSCRIPTION_ID = ""

os.environ["AZURE_SUBSCRIPTION_ID"] = SUBSCRIPTION_ID
os.environ["AZURE_RESOURCE_GROUP"]  = "CIS-5270"
os.environ["AZURE_AOAI_ACCOUNT"]    = RESOURCE_GROUP
os.environ["AZURE_OPENAI_API_KEY"]  = OPENAI_API_KEY
os.environ["AZURE_OPENAI_ENDPOINT"] = OPENAI_ENDPOINT

CREDENTIAL = DefaultAzureCredential()

openai_client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    azure_endpoint=OPENAI_ENDPOINT,
    api_version="2025-04-01-preview",
)

TEACHER_DEPLOYMENT = "gpt-4.1-mini"

print("Connected to Azure OpenAI")

Connected to Azure OpenAI


In [2]:
# load the 3k training pool (same pool used for c1/c3)
train_pool = []
with open("data/generated/train_pool_3k.jsonl") as f:
    for line in f:
        train_pool.append(json.loads(line))

print(f"loaded {len(train_pool)} examples")
print(Counter(ex["label"] for ex in train_pool))

loaded 3000 examples
Counter({'NOT MENTIONED': 1000, 'SUPPORTED': 1000, 'CONTRADICTED': 1000})


---
## Config 5 — Evidence-First Justification

Teacher must open by quoting or closely paraphrasing the most relevant passage fragment, then draw the verdict from it. Forces grounding before reasoning — targets hallucinated-evidence failures.

In [3]:
EF_SYSTEM = """You are a fact-checking assistant. Given a passage and a claim, write a one-sentence justification using this structure:
1. Quote or closely paraphrase the most relevant part of the passage.
2. State how that evidence supports the verdict.

Format your full response as: LABEL: justification sentence
Label must be one of: SUPPORTED, CONTRADICTED, NOT MENTIONED

The justification must lead with the evidence, not the conclusion."""


def get_justification_ef(passage, claim, label):
    user_msg = f"Passage: {passage}\n\nClaim: {claim}\n\nVerdict: {label}"
    for attempt in range(3):
        try:
            resp = openai_client.chat.completions.create(
                model=TEACHER_DEPLOYMENT,
                messages=[
                    {"role": "system", "content": EF_SYSTEM},
                    {"role": "user",   "content": user_msg},
                ],
                temperature=0.3,
                max_tokens=150,
            )
            text = resp.choices[0].message.content.strip()
            for lbl in ["SUPPORTED", "CONTRADICTED", "NOT MENTIONED"]:
                if text.upper().startswith(lbl + ":"):
                    text = text[len(lbl)+1:].strip()
                    break
            return text
        except Exception as e:
            print(f"  attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)
    return None

In [4]:
# spot-check before full run
random.seed(5)
for ex in random.sample(train_pool, 3):
    result = get_justification_ef(ex["passage"], ex["claim"], ex["label"])
    print(f"label:   {ex['label']}")
    print(f"claim:   {ex['claim']}")
    print(f"passage: {ex['passage'][:100]}")
    print(f"result:  {result}")
    print()

label:   SUPPORTED
claim:   Kirk Douglas acted in Lonely Are the Brave.
passage: He produced and starred in Lonely Are the Brave -LRB- 1962 -RRB- , considered a cult classic , and S
result:  The passage states, "He produced and starred in Lonely Are the Brave," directly indicating that Kirk Douglas acted in the film.

label:   CONTRADICTED
claim:   Measles develop exactly 11 days after exposure to an infected person.
passage: Symptoms usually develop 10 -- 12 days after exposure to an infected person and last 7 -- 10 days .
result:  The passage states that symptoms usually develop 10 -- 12 days after exposure, indicating a range rather than exactly 11 days.

label:   CONTRADICTED
claim:   Vikram closed a welfare association.
passage: He has been a brand ambassador of Sanjeevani Trust and a school for special children , Vidya Sudha ,
result:  The passage states that Vikram is "running his own welfare association through the Vikram Foundation," which contradicts the claim that he closed 

In [5]:
EF_OUT = "data/generated/sft_data_ef.jsonl"

done_ids = set()
if os.path.exists(EF_OUT):
    with open(EF_OUT) as f:
        for line in f:
            done_ids.add(json.loads(line)["id"])
    print(f"resuming, {len(done_ids)} already done")

skipped = 0
with open(EF_OUT, "a") as out_f:
    for i, ex in enumerate(train_pool):
        if ex["id"] in done_ids:
            continue

        justification = get_justification_ef(ex["passage"], ex["claim"], ex["label"])
        if justification is None:
            skipped += 1
            continue

        record = {
            "id": ex["id"],
            "messages": [
                {"role": "system",    "content": EF_SYSTEM},
                {"role": "user",      "content": f"Passage: {ex['passage']}\n\nClaim: {ex['claim']}"},
                {"role": "assistant", "content": f"{ex['label']}: {justification}"},
            ],
            "label": ex["label"],
            "justification": justification,
            "passage": ex["passage"],
            "claim": ex["claim"],
        }
        out_f.write(json.dumps(record) + "\n")
        out_f.flush()

        if (i + 1) % 500 == 0:
            print(f"[{i+1}/{len(train_pool)}] skipped={skipped}")

print(f"done. skipped {skipped}")

  attempt 1 failed: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': True, 'severity': 'medium'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}
[500/3000] skipped=0
[1000/3000] skipped=0
[1500/3000] skipped=0
[2000/3000] skipped=0
[2500/3000] skipped=0
[3000/3000] skipped=0
done. skipped 0


In [6]:
ef_data = [json.loads(l) for l in open(EF_OUT)]
print(f"total EF examples: {len(ef_data)}")
print(Counter(ex["label"] for ex in ef_data))
print("\nsample:")
print(json.dumps(ef_data[0]["messages"], indent=2))

total EF examples: 3000
Counter({'NOT MENTIONED': 1000, 'SUPPORTED': 1000, 'CONTRADICTED': 1000})

sample:
[
  {
    "role": "system",
    "content": "You are a fact-checking assistant. Given a passage and a claim, write a one-sentence justification using this structure:\n1. Quote or closely paraphrase the most relevant part of the passage.\n2. State how that evidence supports the verdict.\n\nFormat your full response as: LABEL: justification sentence\nLabel must be one of: SUPPORTED, CONTRADICTED, NOT MENTIONED\n\nThe justification must lead with the evidence, not the conclusion."
  },
  {
    "role": "user",
    "content": "Passage: The house gained local notoriety in 1942 when owner Reuben Stoller was found dead in its basement ; his murder was never solved .\n\nClaim: George Best played as a linebacker."
  },
  {
    "role": "assistant",
    "content": "NOT MENTIONED: The passage discusses the notoriety of a house due to Reuben Stoller's unsolved murder in 1942 but does not mention

In [21]:
# strip metadata for upload
with open(EF_OUT) as f_in, open("data/generated/sft_data_ef_upload.jsonl", "w") as f_out:
    for line in f_in:
        ex = json.loads(line)
        f_out.write(json.dumps({"messages": ex["messages"]}) + "\n")
print("done → data/generated/sft_data_ef_upload.jsonl")

done → data/generated/sft_data_ef_upload.jsonl


---
## Config 6 — Contrastive Justification

Teacher eliminates the two wrong labels in one clause, then states the correct one. Concise by design — single sentence, two parts: ruling out + conclusion. Targets SUPPORTS/REFUTES confusion.

In [12]:
CT_SYSTEM = """You are a fact-checking assistant. Given a passage and a claim, write a one-sentence justification that explicitly names all three labels and explains why the correct one applies and the other two do not.

Format: LABEL: justification sentence
Label must be one of: SUPPORTED, CONTRADICTED, NOT MENTIONED
Example structure: "While the claim is not [wrong_label_1] because [reason] nor [wrong_label_2] because [reason], it is [correct_label] because [evidence]." """


def get_justification_ct(passage, claim, label):
    user_msg = f"Passage: {passage}\n\nClaim: {claim}\n\nVerdict: {label}"
    for attempt in range(3):
        try:
            resp = openai_client.chat.completions.create(
                model=TEACHER_DEPLOYMENT,
                messages=[
                    {"role": "system", "content": CT_SYSTEM},
                    {"role": "user",   "content": user_msg},
                ],
                temperature=0.3,
                max_tokens=150,
            )
            text = resp.choices[0].message.content.strip()
            for lbl in ["SUPPORTED", "CONTRADICTED", "NOT MENTIONED"]:
                if text.upper().startswith(lbl + ":"):
                    text = text[len(lbl)+1:].strip()
                    break
            return text
        except Exception as e:
            print(f"  attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)
    return None

In [10]:
# spot-check before full run
random.seed(6)
for ex in random.sample(train_pool, 3):
    result = get_justification_ct(ex["passage"], ex["claim"], ex["label"])
    print(f"label:   {ex['label']}")
    print(f"claim:   {ex['claim']}")
    print(f"passage: {ex['passage'][:100]}")
    print(f"result:  {result}")
    print()

label:   SUPPORTED
claim:   Rashida Jones was in a movie.
passage: She has had film roles in I Love You , Man -LRB- 2009 -RRB- ; Our Idiot Brother -LRB- 2011 -RRB- ; T
result:  While the claim is not CONTRADICTED because the passage explicitly lists movies she was in, nor NOT MENTIONED because the passage clearly states her film roles, it is SUPPORTED because it directly confirms Rashida Jones appeared in multiple movies.

label:   SUPPORTED
claim:   From 1933 to 1945, Adolf Hitler was Chancellor of Germany.
passage: Adolf Hitler -LRB- -LSB- ˈadɔlf ˈhɪtlɐ -RSB- ; 20 April 1889 -- 30 April 1945 -RRB- was a German pol
result:  The claim is not CONTRADICTED because the passage explicitly states Hitler was Chancellor of Germany from 1933 to 1945, nor is it NOT MENTIONED because the passage clearly includes this information; therefore, it is SUPPORTED.

label:   CONTRADICTED
claim:   The Weeknd is not a singer.
passage: The Weeknd -LRB- born Abel Makkonen Tesfaye ; February 16 , 1990 -RRB- 

In [14]:
CT_OUT = "data/generated/sft_data_ct.jsonl"

done_ids = set()
if os.path.exists(CT_OUT):
    with open(CT_OUT) as f:
        for line in f:
            done_ids.add(json.loads(line)["id"])
    print(f"resuming, {len(done_ids)} already done")

skipped = 0
with open(CT_OUT, "a") as out_f:
    for i, ex in enumerate(train_pool):
        if ex["id"] in done_ids:
            continue

        justification = get_justification_ct(ex["passage"], ex["claim"], ex["label"])
        if justification is None:
            skipped += 1
            continue

        record = {
            "id": ex["id"],
            "messages": [
                {"role": "system",    "content": CT_SYSTEM},
                {"role": "user",      "content": f"Passage: {ex['passage']}\n\nClaim: {ex['claim']}"},
                {"role": "assistant", "content": f"{ex['label']}: {justification}"},
            ],
            "label": ex["label"],
            "justification": justification,
            "passage": ex["passage"],
            "claim": ex["claim"],
        }
        out_f.write(json.dumps(record) + "\n")
        out_f.flush()

        if (i + 1) % 500 == 0:
            print(f"[{i+1}/{len(train_pool)}] skipped={skipped}")

print(f"done. skipped {skipped}")

resuming, 456 already done
[500/3000] skipped=0
[1000/3000] skipped=0
[1500/3000] skipped=0
[2000/3000] skipped=0
[2500/3000] skipped=0
[3000/3000] skipped=0
done. skipped 0


In [15]:
ct_data = [json.loads(l) for l in open(CT_OUT)]
print(f"total CT examples: {len(ct_data)}")
print(Counter(ex["label"] for ex in ct_data))
print("\nsample:")
print(json.dumps(ct_data[0]["messages"], indent=2))

total CT examples: 3000
Counter({'NOT MENTIONED': 1000, 'SUPPORTED': 1000, 'CONTRADICTED': 1000})

sample:
[
  {
    "role": "system",
    "content": "You are a fact-checking assistant. Given a passage and a claim, write a one-sentence justification that explicitly names all three labels and explains why the correct one applies and the other two do not.\n\nFormat: LABEL: justification sentence\nLabel must be one of: SUPPORTED, CONTRADICTED, NOT MENTIONED\nExample structure: \"While the claim is not [wrong_label_1] because [reason] nor [wrong_label_2] because [reason], it is [correct_label] because [evidence].\" "
  },
  {
    "role": "user",
    "content": "Passage: The house gained local notoriety in 1942 when owner Reuben Stoller was found dead in its basement ; his murder was never solved .\n\nClaim: George Best played as a linebacker."
  },
  {
    "role": "assistant",
    "content": "NOT MENTIONED: While the claim is not SUPPORTED because the passage does not mention George Best o

In [22]:
# strip metadata for upload
with open(CT_OUT) as f_in, open("data/generated/sft_data_ct_upload.jsonl", "w") as f_out:
    for line in f_in:
        ex = json.loads(line)
        f_out.write(json.dumps({"messages": ex["messages"]}) + "\n")
print("done → data/generated/sft_data_ct_upload.jsonl")

done → data/generated/sft_data_ct_upload.jsonl


---
## Validation Files

Generate matching validation sets for each style using `sft_val.jsonl` as the example pool (same 300 examples, re-labelled with the new system prompt).

In [16]:
# load the original val pool (raw examples, not the generated justifications)
val_pool = []
with open("data/joined/fever_dev_joined.jsonl") as f:
    for line in f:
        val_pool.append(json.loads(line))

# get the same 300 IDs used in sft_val.jsonl
val_ids = set()
with open("data/generated/sft_val.jsonl") as f:
    for line in f:
        val_ids.add(json.loads(line)["id"])

val_pool = [ex for ex in val_pool if ex["id"] in val_ids]
print(f"val pool: {len(val_pool)} examples")
print(Counter(ex["label"] for ex in val_pool))

val pool: 300 examples
Counter({'CONTRADICTED': 100, 'SUPPORTED': 100, 'NOT MENTIONED': 100})


In [17]:
# evidence-first val
EF_VAL_OUT = "data/generated/sft_val_ef.jsonl"

done_ids = set()
if os.path.exists(EF_VAL_OUT):
    with open(EF_VAL_OUT) as f:
        for line in f:
            done_ids.add(json.loads(line)["id"])
    print(f"resuming, {len(done_ids)} already done")

skipped = 0
with open(EF_VAL_OUT, "a") as out_f:
    for ex in val_pool:
        if ex["id"] in done_ids:
            continue
        justification = get_justification_ef(ex["passage"], ex["claim"], ex["label"])
        if justification is None:
            skipped += 1
            continue
        record = {
            "id": ex["id"],
            "messages": [
                {"role": "system",    "content": EF_SYSTEM},
                {"role": "user",      "content": f"Passage: {ex['passage']}\n\nClaim: {ex['claim']}"},
                {"role": "assistant", "content": f"{ex['label']}: {justification}"},
            ],
            "label": ex["label"],
        }
        out_f.write(json.dumps(record) + "\n")
        out_f.flush()

print(f"EF val done. skipped {skipped} → {EF_VAL_OUT}")

EF val done. skipped 0 → data/generated/sft_val_ef.jsonl


In [18]:
# strip metadata for upload
with open(EF_VAL_OUT) as f_in, open("data/generated/sft_val_ef_upload.jsonl", "w") as f_out:
    for line in f_in:
        ex = json.loads(line)
        f_out.write(json.dumps({"messages": ex["messages"]}) + "\n")
print("done")

done


In [19]:
# contrastive val
CT_VAL_OUT = "data/generated/sft_val_ct.jsonl"

done_ids = set()
if os.path.exists(CT_VAL_OUT):
    with open(CT_VAL_OUT) as f:
        for line in f:
            done_ids.add(json.loads(line)["id"])
    print(f"resuming, {len(done_ids)} already done")

skipped = 0
with open(CT_VAL_OUT, "a") as out_f:
    for ex in val_pool:
        if ex["id"] in done_ids:
            continue
        justification = get_justification_ct(ex["passage"], ex["claim"], ex["label"])
        if justification is None:
            skipped += 1
            continue
        record = {
            "id": ex["id"],
            "messages": [
                {"role": "system",    "content": CT_SYSTEM},
                {"role": "user",      "content": f"Passage: {ex['passage']}\n\nClaim: {ex['claim']}"},
                {"role": "assistant", "content": f"{ex['label']}: {justification}"},
            ],
            "label": ex["label"],
        }
        out_f.write(json.dumps(record) + "\n")
        out_f.flush()

print(f"CT val done. skipped {skipped} → {CT_VAL_OUT}")

CT val done. skipped 0 → data/generated/sft_val_ct.jsonl


In [20]:
# strip metadata for upload
with open(CT_VAL_OUT) as f_in, open("data/generated/sft_val_ct_upload.jsonl", "w") as f_out:
    for line in f_in:
        ex = json.loads(line)
        f_out.write(json.dumps({"messages": ex["messages"]}) + "\n")
print("done")

done
